# Diffusion Models for Protein Design
## DDPM · Score Matching · RFdiffusion · FrameDiff · Genie2

**TL;DR:** Diffusion models learn to generate data by learning to reverse a noise process. Start with a real protein structure, gradually add Gaussian noise until it's pure noise, then train a neural network to reverse this process (denoise). At inference time, start from pure noise and denoise step-by-step to generate a new protein. AlphaFold3 uses diffusion for its final structure generation module. RFdiffusion (Baker lab) can design novel protein binders from scratch.

**After this notebook you can:**
- Implement DDPM (Denoising Diffusion Probabilistic Model) forward and reverse processes
- Understand score matching and the connection to diffusion
- Explain how RFdiffusion generates protein backbones
- Implement a simple protein backbone diffusion model
- Connect diffusion to AF3's structure module (replaces direct coordinate regression)

## Diffusion Models — Concepts for Beginners

| Term | Plain English |
|------|--------------|
| **diffusion (forward process)** | Gradually add Gaussian noise to data over T steps until it becomes pure noise — a Markov chain |
| **denoising (reverse process)** | Neural network learns to undo noise one step at a time — "what was the cleaner version of this?" |
| **DDPM** | Denoising Diffusion Probabilistic Model — the standard framework (Ho et al. 2020) |
| **noise schedule β_t** | Controls how much noise is added at each step; starts small, grows to completely corrupt the signal |
| **ᾱ_t (alpha bar)** | Cumulative noise product: x_t = √ᾱ_t · x₀ + √(1-ᾱ_t) · ε — lets you jump to any noise level directly |
| **ε-prediction** | Model predicts the noise ε that was added, not the clean signal directly — more stable to train |
| **score function** | Gradient of log-probability: ∇_x log p(x) — points toward higher-density regions; used in score-based models |
| **UNet** | Common architecture for diffusion: encodes then decodes with skip connections — predicts noise at each resolution |
| **denoising trajectory** | The sequence of intermediate structures from noise to final protein structure |
| **classifier-free guidance** | Train with/without condition labels, combine both at inference to control generation strength |
| **RFdiffusion** | ProteinMPNN + diffusion — generates protein backbone structures from scratch (Watson et al. 2023) |
| **inpainting** | Generate a new region of structure while keeping the rest fixed — used for protein loop design |

## Beginner Teaching Frame

**Who should fully work through this notebook:** advanced students or curious students on a concept-first pass.

**How to study it on a first pass:**
- learn the forward process, reverse process, and denoising idea
- understand why diffusion is attractive for protein generation
- leave the deeper math for a second pass if needed

**Common traps:** trying to memorize equations without understanding what the noising and denoising processes are doing conceptually.


## Canonical Resource Upgrade

Strong sequence for learning:
- [Lilian Weng on Diffusion Models](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/)
- [Annotated Diffusion](https://huggingface.co/blog/annotated-diffusion)
- [DDPM paper](https://arxiv.org/abs/2006.11239)


## Prerequisites & Learning Path

| | |
|--|--|
| Prerequisites | 07/01 AF3 Architecture (rigid frames), 06/02 GNNs (equivariance), 00/08 Math Foundations |
| You Are Here | Module 12/01 — Diffusion Models (generative models for protein design) |
| Next Steps | 10/01 Fine-tuning Capstone, 11/01 Membrane Protein Dynamics |
| Goal | Implement DDPM, understand protein diffusion, connect to RFdiffusion and AF3 |

### From Scratch? Start Here:
1. [Lilian Weng blog — Diffusion Models](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/) — best written intro
2. [Andrej Karpathy — Makemore: Diffusion intuition](https://www.youtube.com/watch?v=TCH_1BHY58I)
3. 00/08 Math Foundations (probability, Gaussian distributions)
4. This notebook

**Time:** 15–20 hours | **Difficulty:** 5/5

### Cross-References
- Builds on: 07/01 (AF3 uses diffusion module), 06/02 (equivariant GNNs used in diffusion backbone)
- Used by: 10/01 (fine-tune a diffusion-based model), 11/01 (conformational dynamics as diffusion)
- Related: 07/04 covers AF3 full-scale training including diffusion module

---
## Section 1 — DDPM Theory and the Forward Process

### Key Concepts

**The Core Idea:** A diffusion model defines two Markov chains:
1. **Forward process** `q` — gradually corrupts data with Gaussian noise over `T` steps until the data becomes indistinguishable from standard normal noise
2. **Reverse process** `p_θ` — a learned neural network that undoes this corruption, step by step

At inference time, you sample pure Gaussian noise and run the reverse process to get a new data sample.

### Mathematical Formulation

**Forward process** (fixed, no learnable parameters):
```
q(x_t | x_{t-1}) = N(x_t; √(1-β_t) x_{t-1}, β_t I)
```
where `β_t` is a small positive constant called the **noise schedule**.

**The closed-form shortcut** — you can sample any `x_t` directly from `x_0` without simulating all intermediate steps:
```
q(x_t | x_0) = N(x_t; √ᾱ_t x_0, (1-ᾱ_t) I)
ᾱ_t = ∏_{s=1}^{t} (1 - β_s)   (cumulative product of alphas)
x_t = √ᾱ_t · x_0 + √(1-ᾱ_t) · ε,   ε ~ N(0, I)
```
This makes training efficient — you can compute the loss at any timestep `t` in a single forward pass.

**Training objective** — the neural network predicts the noise `ε` that was added:
```
L = E_{x_0, ε, t}[ || ε - ε_θ(√ᾱ_t x_0 + √(1-ᾱ_t) ε, t) ||² ]
```
This is just **MSE between true noise and predicted noise** — surprisingly simple!

**Why predict noise rather than `x_0` directly?**  
Empirically, noise prediction gives better sample quality. Mathematically, it corresponds to learning the score function `∇ log p(x_t)`, which has better gradient properties (see Section 4).

### 📦 Libraries Used Here

- **`torch`** — Deep learning library — builds and trains neural networks.
- **`numpy as np`** — Fast array maths. np.linspace(0,1,1000) = 1000 evenly-spaced numbers from 0 to 1.
- **`matplotlib.pyplot as plt`** — Plotting library. plt.plot(x, y) draws a line; plt.show() displays it.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# DDPM Theory — forward process
print("DDPM Theory: Forward Process")
print("=" * 50)

class DDPMForward:
"""DDPM forward (noising) process as a pure Python class.

Implements the Markov chain q(x_t | x_{t-1}) = N(x_t; √(1-β_t)·x_{t-1}, β_t·I)
and the closed-form jump q(x_t | x_0) = N(x_t; √ᾱ_t·x_0, (1-ᾱ_t)·I)."""
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02):
        """Set up the noise schedule.

Args:
    T (int): total number of diffusion timesteps (default 1000).
    beta_start (float): initial noise level β_1 (default 1e-4).
    beta_end (float): final noise level β_T (default 0.02)."""
        self.T = T
        self.betas = torch.linspace(beta_start, beta_end, T)
        self.alphas = 1 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)

    def q_sample(self, x0, t):
        """x_t = sqrt(ᾱ_t)*x0 + sqrt(1-ᾱ_t)*ε"""
        sqrt_ab = self.alpha_bar[t].sqrt()
        sqrt_1_ab = (1 - self.alpha_bar[t]).sqrt()
        eps = torch.randn_like(x0)
        return sqrt_ab * x0 + sqrt_1_ab * eps, eps

ddpm = DDPMForward(T=1000)

# Visualize signal-to-noise ratio
timesteps = [0, 100, 250, 500, 750, 999]
print("Signal retention at each timestep:")
for t in timesteps:
    snr = ddpm.alpha_bar[t].item()
    print(f"  t={t:4d}: signal={snr:.4f}, noise={1-snr:.4f}")

# Demonstrate forward process on toy 1D data
x0 = torch.tensor([3.0])
print()
for t in [0, 100, 500, 999]:
    xt, eps = ddpm.q_sample(x0.unsqueeze(0), t)
    print(f"  t={t:4d}: x0={x0.item():.2f} → x_t={xt.item():.2f}")
print()
print("Key: at t=999, x_t ≈ pure Gaussian noise regardless of x0")
print("Training: predict eps from x_t → enables denoising")

> **Expected output:** DDPM forward process: signal retention values decreasing from ~1.0 at t=0 to ~0.0 at t=999, showing how noise progressively destroys the signal  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

---
## Section 2 — Simple DDPM Implementation (2D Toy Example)

### Architecture: U-Net Denoiser

For images, DDPM uses a **U-Net** with:
- Encoder path (downsampling with strided convolutions)
- Skip connections between encoder and decoder
- Decoder path (upsampling)
- **Timestep conditioning** via sinusoidal embeddings injected at each resolution

For proteins, the denoiser is an **equivariant GNN** (like the IPA in AF3) that respects SE(3) symmetry — rotating/translating the protein should give the same score, just rotated/translated.

### Why Sinusoidal Time Embeddings?

The network needs to know what noise level to denoise at. We encode the integer timestep `t` as a sinusoidal vector (identical to transformer positional encodings):
```
PE(t, 2i)   = sin(t / 10000^(2i/d))
PE(t, 2i+1) = cos(t / 10000^(2i/d))
```
This gives the network a continuous, smooth representation of the noise level.

### Training Algorithm

```
repeat:
  1. Sample x_0 from training data
  2. Sample t uniformly from {1, ..., T}
  3. Sample ε ~ N(0, I)
  4. Compute x_t = √ᾱ_t · x_0 + √(1-ᾱ_t) · ε  (closed form!)
  5. Compute loss = || ε - ε_θ(x_t, t) ||²
  6. Gradient step on θ
```

### 📖 Reading Guide — DDPM Forward Process — Adding Noise

`beta = torch.linspace(beta_start, beta_end, T)`
→ *Noise schedule: start with almost no noise (beta≈0.0001) and gradually add more (beta≈0.02) over T=1000 steps.*

`alpha = 1 - beta`
→ *alpha[t] = how much of the original signal survives at step t. alpha close to 1 = mostly signal.*

`alpha_bar = torch.cumprod(alpha, dim=0)`
→ *Cumulative product: alpha_bar[t] = alpha[0] × alpha[1] × ... × alpha[t]. This is the total signal remaining after t steps.*

`def q_sample(x0, t, noise=None):`
→ *Forward process: add noise to a clean sample x0 at time step t.*

`sqrt_alpha_bar_t = alpha_bar[t].sqrt()`
→ *Scale factor for the clean signal. As t increases, this approaches 0 (signal fades).*

`sqrt_one_minus * noise`
→ *Scale factor for the noise. As t increases, this approaches 1 (pure noise).*

`return sqrt_alpha_bar_t * x0 + sqrt_one_minus * noise`
→ *Noisy sample = scaled clean + scaled noise. At t=T, x_T is nearly pure Gaussian noise.*



In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Simple DDPM on 2D toy data
class NoisePredictor(nn.Module):
    """Simple MLP noise predictor for 2D data."""
    def __init__(self, data_dim=2, time_embed_dim=32, hidden=128):
        """Build the UNet-style noise prediction network.

Args:
    data_dim (int): dimensionality of the data space (default 2 for toy 2D).
    time_embed_dim (int): sinusoidal time embedding dimension (default 32).
    hidden (int): hidden layer width for the MLP backbone (default 128)."""
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(1, time_embed_dim), nn.SiLU(),
            nn.Linear(time_embed_dim, time_embed_dim)
        )
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_embed_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, data_dim)
        )

    def forward(self, x, t):
        """Predict the noise added to x at timestep t.

Args:
    x (Tensor): noisy data of shape (B, data_dim).
    t (Tensor): integer timestep indices of shape (B,).

Returns:
    Tensor: predicted noise ε of shape (B, data_dim)."""
        t_emb = self.time_embed(t.float().unsqueeze(-1) / 1000)
        return self.net(torch.cat([x, t_emb], dim=-1))

class DDPMTrainer:
"""Wraps the forward process and computes the DDPM training loss.

The loss is E[||ε − ε_θ(x_t, t)||²] — mean squared error between
the true noise and the model's noise prediction."""
    def __init__(self, T=1000):
        """Initialise trainer and precompute ᾱ_t schedule.

Args:
    T (int): total number of diffusion timesteps."""
        self.T = T
        betas = torch.linspace(1e-4, 0.02, T)
        self.alpha_bar = torch.cumprod(1 - betas, dim=0)

    def loss(self, model, x0):
        """Compute one DDPM training step loss.

Samples random timestep t and noise ε, forms x_t, then asks model to
predict ε.  Returns MSE between predicted and true noise.

Args:
    model (nn.Module): noise prediction network ε_θ.
    x0 (Tensor): clean data batch of shape (B, data_dim).

Returns:
    Tensor: scalar MSE loss."""
        B = x0.shape[0]
        t = torch.randint(0, self.T, (B,))
        ab = self.alpha_bar[t].unsqueeze(-1)
        eps = torch.randn_like(x0)
        x_t = ab.sqrt() * x0 + (1-ab).sqrt() * eps
        eps_pred = model(x_t, t)
        return nn.MSELoss()(eps_pred, eps)

# Train on 2D Swiss Roll
torch.manual_seed(42)
from sklearn.datasets import make_swiss_roll
X, _ = make_swiss_roll(n_samples=2000, noise=0.1)
X = torch.tensor(X[:, [0, 2]], dtype=torch.float32) / 5.0  # use 2 of 3 dims

model = NoisePredictor(data_dim=2)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
trainer = DDPMTrainer(T=100)

for step in range(200):
    idx = torch.randint(0, len(X), (64,))
    loss = trainer.loss(model, X[idx])
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step % 50 == 0:
        print(f"Step {step}: loss={loss.item():.4f}")

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print("DDPM objective: predict noise eps from noisy x_t at timestep t")

---
## Section 3 — Protein Backbone Diffusion (FrameDiff / RFdiffusion Concepts)

### Why Proteins Are Different From Images

Image diffusion operates on pixel arrays `x ∈ ℝ^(H×W×C)` — flat Euclidean space. You can add Gaussian noise directly and it stays in valid pixel space.

**Protein backbones are sequences of rigid frames.** Each residue's backbone is defined by:
- A **translation** `t ∈ ℝ³` (Cα position)
- A **rotation** `R ∈ SO(3)` (orientation of the local frame, defined by the N-Cα-C plane)

The rotation group SO(3) is a **curved manifold** — it is NOT flat Euclidean space. Adding standard Gaussian noise to a rotation matrix would produce a matrix that is no longer in SO(3) (i.e., no longer a valid rotation). You would need to project back, which distorts the geometry.

### SO(3) Diffusion

The solution is to define diffusion **intrinsically on the manifold** SO(3):
- Forward process: apply small random rotations at each step (using the exponential map)
- Noise schedule: parameterized by the variance of the wrapped Gaussian on SO(3)
- At high noise: uniform distribution on SO(3) (like pure Gaussian noise in Euclidean space)
- Reverse process: denoise using the Riemannian score `∇_{SO(3)} log p(R_t)`

### RFdiffusion Architecture

RFdiffusion (Watson et al. 2023) builds on RoseTTAFold:
1. Input: noisy backbone frames at timestep `t`, conditioning information (target structure, symmetry)
2. Backbone: RoseTTAFold network (track attention over sequence, pair, and structure)
3. Output: predicted noise in SE(3) space (both SO(3) rotation score and ℝ³ translation score)
4. Conditioning: partial diffusion (fix part of structure, diffuse the rest) enables binder design

### AF3 Diffusion Module

AlphaFold3 uses diffusion differently from RFdiffusion:
- AF3 first runs the Pairformer to get rich sequence/pair representations
- Then uses **atom-level diffusion** conditioned on those representations
- Diffuses over all atom positions (not just Cα)
- The denoiser is conditioned on Pairformer outputs at every step
- This replaces AF2's Structure Module (direct coordinate regression via IPA)

Key advantage: diffusion allows AF3 to represent **multimodal distributions** over structures, rather than collapsing to a single prediction.

### 📖 Reading Guide — Noise Predictor Network (U-Net style)

`class NoisePredictor(nn.Module):`
→ *Neural network that learns to predict the noise added at each step. During generation, we run it in reverse.*

`self.time_embed = nn.Embedding(T, time_dim)`
→ *Time embedding: convert the integer timestep t into a vector. The network needs to know 'how noisy is this input?'*

`self.net = nn.Sequential(...)`
→ *The denoising network: noisy_input + time_embedding → predicted_noise.*

`def forward(self, x_t, t):`
→ *Given noisy input x_t at timestep t, predict the noise ε that was added.*

`t_emb = self.time_embed(t)`
→ *Look up the time embedding for this timestep — like a 'noise level indicator'.*

`return self.net(torch.cat([x_t, t_emb], dim=-1))`
→ *Concatenate noisy input with time embedding, then pass through the network to predict noise.*



In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Protein backbone diffusion concepts (RFdiffusion/FrameDiff)
print("Protein Backbone Diffusion: Key Concepts")
print("=" * 60)
print()
print("Standard DDPM (Euclidean): diffuse xyz coordinates")
print("  x_t = sqrt(ᾱ_t)*x0 + sqrt(1-ᾱ_t)*ε  where ε ~ N(0,I)")
print()
print("SE(3) Diffusion (backbone frames):")
print("  Frames T = (R, x) ∈ SE(3) where R ∈ SO(3), x ∈ ℝ³")
print("  Translation: same as DDPM on xyz")
print("  Rotation:    IGSO3 distribution (Gaussian on SO(3) manifold)")
print()
print("RFdiffusion approach:")
print("  1. Condition on partial structure (motif scaffolding)")
print("  2. Diffuse backbone frames (N CA C O)")
print("  3. Trained on PDB structures with self-conditioning")
print("  4. ~40M parameters, trained on 58k PDB proteins")
print()

# Simplified frame diffusion (translations only)
class FrameTranslationDiffusion(nn.Module):
    """Simplified diffusion on CA coordinates (translation part of SE(3))."""
    def __init__(self, T=200, c_z=64):
        """Initialise protein backbone diffusion module.

Args:
    T (int): number of diffusion timesteps for backbone coordinates.
    c_z (int): pair representation dimension from the structure encoder."""
        super().__init__()
        self.T = T
        betas = torch.linspace(1e-4, 0.01, T)
        self.alpha_bar = torch.cumprod(1 - betas, dim=0)
        # Denoiser: uses pair representation z
        self.denoiser = nn.Sequential(
            nn.Linear(3 + c_z, 128), nn.ReLU(),
            nn.Linear(128, 3)  # predict coords
        )

    def denoise(self, x_t, z, t):
        """Predict x0 from x_t given pair repr z and timestep t."""
        L = x_t.shape[0]
        z_mean = z.mean(dim=1)  # (L, c_z) — mean over j
        inp = torch.cat([x_t, z_mean], dim=-1)
        return self.denoiser(inp)

torch.manual_seed(42)
L, c_z = 20, 64
model = FrameTranslationDiffusion(T=200, c_z=c_z)
x_t = torch.randn(L, 3) * 10
z = torch.randn(L, L, c_z)
x0_pred = model.denoise(x_t, z, t=100)
print(f"Denoiser: x_t {x_t.shape} + z {z.shape} → x0_pred {x0_pred.shape}")

---
## Section 4 — Score Matching and DDIM Sampling

### Score Matching: The Deeper Theory

**What is the score function?**
```
s(x) = ∇_x log p(x)
```
The score is the gradient of the log-probability density with respect to the data. It points toward regions of **higher probability** — like a compass pointing toward where data is more likely.

**Key insight (Song & Ermon 2019):**
```
ε_θ(x_t, t) ≈ -√(1-ᾱ_t) · ∇_{x_t} log q(x_t)
```
Training DDPM (predicting noise) is **equivalent to learning the score function** at all noise levels simultaneously. The two objectives are mathematically equivalent up to a scaling factor.

**Langevin dynamics** uses the score to sample:
```
x_{t+1} = x_t + η · ∇ log p(x_t) + √(2η) · ε
```
This is exactly what DDPM's reverse process does — Langevin dynamics guided by the learned score.

### DDIM: Deterministic Faster Sampling

DDPM requires T=1000 reverse steps for high-quality samples. **DDIM (Song et al. 2020)** introduces a deterministic reverse process:
```
x_{t-1} = √ᾱ_{t-1} · [(x_t - √(1-ᾱ_t)·ε_θ) / √ᾱ_t] + √(1-ᾱ_{t-1}) · ε_θ
             ↑ predicted x_0                                  ↑ direction toward x_t
```
Since the reverse process is deterministic, you can **skip timesteps** — sample at t={1000, 980, 960, ...} using only 50 steps. This gives 20x speedup with similar quality.

### Classifier-Free Guidance (CFG)

For conditional generation (e.g., design a protein that binds target X):
```
ε_guided(x_t, c) = ε_θ(x_t, ∅) + w · (ε_θ(x_t, c) - ε_θ(x_t, ∅))
```
- Train with condition `c` (binding target) AND without (`c=∅`, unconditional)
- At inference, combine both to amplify the conditional signal
- `w > 1`: stronger guidance toward condition (more specific, less diverse)
- `w = 1`: standard conditional generation
- `w = 0`: unconditional generation

RFdiffusion uses this to design binders: condition on a target protein structure, guide generation toward structures that are physically compatible with the target.

### 📖 Reading Guide — DDPM Reverse Process — Denoising (p_sample)

`def p_sample(model, x_t, t):`
→ *One denoising step: take noisy x_t at time t and produce slightly less noisy x_{t-1}.*

`eps_theta = model(x_t, t_tensor)`
→ *Ask the network: 'what noise do you think was added?' The answer guides denoising.*

`x0_pred = (x_t - sqrt_one_minus * eps_theta) / sqrt_alpha_bar`
→ *Estimate the clean signal from the noisy version using the predicted noise.*

`mean = sqrt_alpha_bar_prev * x0_pred + sqrt_one_minus_prev * eps_theta`
→ *Compute the mean of the reverse distribution p(x_{t-1} | x_t).*

`return mean + sigma * torch.randn_like(x_t) if t > 0 else mean`
→ *Add a tiny amount of new noise (sigma) — except at t=0 where we return the clean sample directly.*



In [ ]:
import torch
import numpy as np

# Score matching + DDIM sampling
print("Score Matching and DDIM")
print("=" * 50)
print()
print("Score matching:")
print("  Score = ∇_x log p(x) — gradient of log-density")
print("  DDPM implicitly learns score: s_θ(x_t,t) = -ε_θ(x_t,t)/sqrt(1-ᾱ_t)")
print()
print("DDIM (Denoising Diffusion Implicit Models):")
print("  Non-Markovian sampling: skip timesteps for faster generation")
print("  10-50 DDIM steps ≈ 1000 DDPM steps in quality")
print()

class DDIMSampler:
    """DDIM sampling for fast inference."""
    def __init__(self, alpha_bar, n_steps=50):
        """Set up DDIM deterministic sampler.

Args:
    alpha_bar (Tensor): precomputed ᾱ_t schedule of shape (T,).
    n_steps (int): number of DDIM denoising steps (<<T, default 50)."""
        self.alpha_bar = alpha_bar
        T = len(alpha_bar)
        # Select subset of timesteps
        step_size = T // n_steps
        self.timesteps = list(range(T-1, 0, -step_size))[:n_steps]

    def step(self, eps_pred, x_t, t, t_prev):
        """DDIM reverse step."""
        ab_t    = self.alpha_bar[t]
        ab_prev = self.alpha_bar[t_prev] if t_prev >= 0 else torch.tensor(1.0)

        # Predicted x0
        x0_pred = (x_t - (1-ab_t).sqrt() * eps_pred) / ab_t.sqrt()
        x0_pred = x0_pred.clamp(-3, 3)

        # Direction pointing to x_t
        dir_xt = (1 - ab_prev).sqrt() * eps_pred

        return ab_prev.sqrt() * x0_pred + dir_xt

# Simulate DDIM sampling
torch.manual_seed(42)
T = 200
betas = torch.linspace(1e-4, 0.01, T)
alpha_bar = torch.cumprod(1 - betas, dim=0)
sampler = DDIMSampler(alpha_bar, n_steps=20)

x = torch.randn(2, 3)  # start from noise
dummy_model = lambda xt, t: xt * 0.1  # dummy denoiser

for i, t in enumerate(sampler.timesteps[:-1]):
    t_prev = sampler.timesteps[i+1]
    eps = dummy_model(x, t)
    x = sampler.step(eps, x, t, t_prev)

print(f"DDIM: {len(sampler.timesteps)} steps instead of {T}")
print(f"Speedup: {T/len(sampler.timesteps):.0f}×")
print(f"Final x: {x.numpy().round(3)}")

---
## Section 5 — Flow Matching: The Method That Replaced Diffusion (2023–2024)

### Why Flow Matching?

Diffusion models add Gaussian noise progressively — but this is arbitrary. **Flow matching** (Lipman et al., 2022) directly learns a vector field v_t(x) that transports samples from noise p_0 to data p_1 along straight(er) paths.

**The key insight:** Instead of learning to denoise (predict ε at each step), learn the velocity field of the ODE:
```
dx/dt = v_t(x),   t ∈ [0, 1]
```
Start at noise x_0 ~ N(0,I), integrate the ODE forward to get a sample x_1 ~ p_data.

**Why better than DDPM?**
- **Straight paths** → fewer NFEs (function evaluations) to generate a sample: 10-20 steps vs 100-1000 for DDPM
- **Simpler training objective**: regress directly onto the interpolation velocity (x_1 - x_0), no noise schedule tuning
- **Exact likelihood** via the continuous normalizing flow (CNF) connection

**Used in:**
- **Boltz-2** (MIT, 2024): biomolecular structure prediction with flow matching
- **Chai-1** (Chai Discovery, 2024): flow matching for protein-ligand complexes
- **AlphaFold3**: diffusion module, conceptually related to flow matching
- **RFdiffusion** v2: uses flow matching over DDPM
- **Stable Diffusion 3 / FLUX**: image generation switched to flow matching

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Flow matching — replaces diffusion (2023-2024)
print("Flow Matching: The Method That Replaced Diffusion")
print("=" * 60)
print()
print("Why flow matching?")
print("  Diffusion: complex SDE training objective")
print("  Flow matching: simpler ODE, straight paths, faster training")
print()
print("Key idea:")
print("  Conditional Flow Matching (CFM): learn vector field u(x,t) s.t.")
print("  dx/dt = u(x,t), mapping noise x~N(0,I) → data x~p_data")
print()
print("Straight flow paths (OT-CFM):")
print("  x_t = (1-t)*x0 + t*x1  (linear interpolation)")
print("  Target velocity: u = x1 - x0 (constant along path!)")
print()

class FlowMatchingModel(nn.Module):
    """Simple flow matching: learn velocity field u_θ(x_t, t) = x1 - x0."""
    def __init__(self, data_dim=2, hidden=128):
        """Build the flow-matching vector field network.

Args:
    data_dim (int): dimensionality of the data space (default 2).
    hidden (int): hidden layer width (default 128)."""
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim + 1, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, data_dim)
        )

    def forward(self, x, t):
        """Predict the noise added to x at timestep t.

Args:
    x (Tensor): noisy data of shape (B, data_dim).
    t (Tensor): integer timestep indices of shape (B,).

Returns:
    Tensor: predicted noise ε of shape (B, data_dim)."""
        t_feat = t.unsqueeze(-1).float()
        return self.net(torch.cat([x, t_feat], dim=-1))

    def fm_loss(self, x0, x1):
        """Conditional flow matching loss."""
        B = x0.shape[0]
        t = torch.rand(B)
        x_t = (1-t.unsqueeze(-1)) * x0 + t.unsqueeze(-1) * x1  # interpolate
        u_target = x1 - x0  # constant velocity
        u_pred = self(x_t, t)
        return nn.MSELoss()(u_pred, u_target)

torch.manual_seed(42)
model = FlowMatchingModel(data_dim=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

x0 = torch.randn(512, 2)         # noise
x1 = torch.randn(512, 2) * 0.5 + torch.tensor([2.0, 0.0])  # data

for step in range(100):
    loss = model.fm_loss(x0, x1)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step % 25 == 0:
        print(f"  Step {step}: FM loss = {loss.item():.4f}")

print("\nFlow matching used in: Boltz-2, AlphaFold3 (diffusion ≈ flow matching limit)")
print("Key advantage: can use ODE solvers (10-100 NFE vs 1000 for DDPM)")

---
## Resources

### 1 — Theory: Foundations and Math
- **DDPM:** Gaussian diffusion, Markov chain reversal, ELBO derivation (Ho et al. 2020)
- **Score Matching:** Stein score, connection to Fisher divergence (Hyvarinen 2005)
- **SE(3) Diffusion:** Geodesics on SO(3), Haar measure, wrapped Gaussians
- **Stochastic Differential Equations:** SDEs as continuous-time diffusion, Ito calculus
- Papers: [DDPM (Ho 2020)](https://arxiv.org/abs/2006.11239), [Score Matching (Song 2021)](https://arxiv.org/abs/2011.13456), [DDIM (Song 2020)](https://arxiv.org/abs/2010.02502)

### 2 — Must-Have Popular Resources
#### 🟢 Start Here (Zero Diffusion Background)
- 🆓 **Lilian Weng Diffusion Blog** — [lilianweng.github.io/posts/2021-07-11-diffusion-models](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/) — 3M reads, start here
- 🆓 **Annotated Diffusion (HuggingFace)** — [huggingface.co/blog/annotated-diffusion](https://huggingface.co/blog/annotated-diffusion) — code + explanation together
- 🆓 **DDPM paper (Ho 2020)** — [arxiv.org/abs/2006.11239](https://arxiv.org/abs/2006.11239) — only 15 pages, readable after the blog
- 🆓 **Ari Seff diffusion explainer** — YouTube — visual, builds intuition before math
- **Blog:** [Lilian Weng — Diffusion Models (2021)](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/) — 3M reads, best written explanation
- **Paper:** [RFdiffusion (Watson 2023 Nature)](https://www.nature.com/articles/s41586-023-06415-8) — Baker lab, 400+ citations
- **Video:** [Diffusion Models from Scratch — Hugging Face](https://huggingface.co/blog/annotated-diffusion) — code walkthrough
- **GitHub:** [RosettaCommons/RFdiffusion](https://github.com/RosettaCommons/RFdiffusion) — 3k+ stars, protein design via diffusion
- **GitHub:** [jasonkyuyim/se3_diffusion](https://github.com/jasonkyuyim/se3_diffusion) — FrameDiff implementation
- **HuggingFace:** [google-deepmind/alphafold3](https://huggingface.co/google-deepmind/alphafold3) — AF3 uses diffusion for structure
- **Kaggle:** [Protein Design Competition 2024](https://www.kaggle.com/competitions)

### 3 — Practicals: Hands-On
- **Exercise 1:** Implement DDPM on MNIST, generate digit images from noise
- **Exercise 2:** Implement DDIM sampling with 50 steps using a pre-trained DDPM
- **Exercise 3:** Run RFdiffusion to design a binding protein for a target structure
- GitHub: [RosettaCommons/RFdiffusion](https://github.com/RosettaCommons/RFdiffusion) — runnable locally or Colab
- HuggingFace: [RFdiffusion Space](https://huggingface.co/spaces/nferruz/ProtGPT2) — run in browser
- Kaggle: [CATH protein structure datasets for training](https://www.kaggle.com/datasets)

### 4 — Real-World Problems
- **computational biology ML teams / Google structural biology research labs:** AF3 final structure module is a diffusion model
- **Baker Lab (UW):** RFdiffusion used to design novel protein binders — validated experimentally
- **drug discovery companies:** Chroma protein design uses diffusion
- **Dataset:** [CATH protein structures](https://huggingface.co/datasets) — 40k protein domains for training
- **Application:** Design a binder for a GPCR target using RFdiffusion + ProteinMPNN sequence design

### 5 — Interview Tips
- **Must know:** What is the forward process? What is the reverse process? What does the neural network learn?
- **Must know:** Why is protein diffusion on SO(3) rather than Euclidean R^3?
- **Must know:** DDIM vs DDPM — what changes, and why is DDIM faster?
- **Common mistake:** Saying diffusion predicts x0 directly — it predicts the noise (or the score)
- **Common mistake:** Not knowing that AF3's structure module uses diffusion (replaces AF2's direct coordinate regression)
- **Pro tip:** Know classifier-free guidance for conditional protein design — the interviewer will ask about conditional generation

### 6 — Hot Industry Topics
- **Trending:** [Genie2](https://github.com/aqlaboratory/genie2) — improved SE(3) diffusion for long proteins
- **Trending:** [Chroma](https://github.com/generatebiomedicines/chroma) — drug discovery companies protein diffusion
- **Trending:** [ProteinSGM](https://github.com/bayeslabs/proteinsgm) — score-based protein generation
- **Trending:** Video diffusion models adapted for MD trajectory generation (MDGen)
- **Build:** Run RFdiffusion to design a symmetric protein cage, evaluate with ESMFold
- **Build:** Implement DDIM on protein backbone Ca coordinates with 50-step sampling

## Interview Q&A — Diffusion Models

**Q: Explain the forward process of DDPM in one equation.**
A: q(x_t | x_0) = N(x_t; √ᾱ_t · x_0, (1-ᾱ_t)I) where ᾱ_t = ∏_{s=1}^t αs, αs = 1-βs. This means: at timestep t, we can directly sample a noisy version of x_0 by adding scaled Gaussian noise. No need to iteratively apply the forward process.

**Q: What does the DDPM network actually predict — x_0 or ε?**
A: The network predicts the noise ε that was added, not x_0 directly. Why? The noise prediction objective is equivalent to score matching (∇_x log p(x_t)), and empirically the noise parameterization gives better training stability and sample quality. x_0 can be recovered: x̂_0 = (x_t - √(1-ᾱ_t)·ε_θ) / √ᾱ_t.

**Q: Why does protein backbone diffusion operate in SO(3) rather than ℝ³?**
A: Protein backbones are rigid frames (N-Cα-C triangle). Diffusing in ℝ³ ignores rotational constraints — you'd generate invalid backbone geometries. SE(3) diffusion treats each residue as a rigid body with position (translation) + orientation (rotation). Rotations live on SO(3), a non-Euclidean manifold. IGSO(3) is the wrapped normal distribution on SO(3).

**Q: What is DDIM and what is its main advantage over DDPM?**
A: DDIM (Denoising Diffusion Implicit Models) uses a non-Markovian forward process that admits deterministic sampling. Instead of 1000 stochastic denoising steps, DDIM can generate samples in 20-50 deterministic steps with comparable quality. The mapping from noise → image is deterministic → same noise = same sample (useful for interpolation, editing).

**Q: What is the connection between diffusion models and score matching?**
A: The score function is s(x) = ∇_x log p(x). Training a diffusion model to predict noise ε is equivalent to training a score network: ε_θ ≈ -√(1-ᾱ_t) · s(x_t). Score matching (Hyvärinen 2005) directly trains this without requiring tractable p(x). This connection (Ho et al. 2020, Song et al. 2020) unified the field.

**Q: Name 3 protein design tools based on diffusion.**
A: (1) RFdiffusion (David Baker lab): de novo backbone design in SE(3), generates diverse scaffolds. (2) FrameDiff / Genie2: academic research models exploring protein backbone manifold. (3) Chroma (drug discovery companies): programmable protein design with text conditioning. (4) AlphaFold3: uses diffusion for all-atom coordinate generation.

### Diffusion Model — Time Complexity
| Operation | Time | Space | Notes |
|-----------|------|-------|-------|
| Forward process q(x_t|x_0) | O(d) | O(d) | Direct sampling — no iteration |
| Training step (predict ε) | O(forward pass) | O(model) | Standard backprop |
| DDPM sampling (T=1000) | O(T × forward) | O(d) | 1000 NFEs |
| DDIM sampling (S=50) | O(S × forward) | O(d) | 50 NFEs |
| SO(3) Euler integration | O(T × n_residues) | O(n) | Per-residue rotation |
| U-Net (image, n pixels) | O(n × channels) | O(n) | Skip connections |

## Mastery Check

On a first pass, success means you can:
1. explain diffusion in plain English
2. explain why denoising can be a generative strategy
3. connect diffusion to protein design rather than image generation only
4. identify what math you would revisit for a deeper pass


---
## Complete DDPM Training + Sampling — Protein Coordinate Generation

This section trains a full DDPM from scratch on 2D protein "shape" data.
You will see: forward process (adding noise) → reverse process (denoising) → sample generation.

### Predict Before Running
> At timestep T=200, what should the structure look like? (Answer: pure Gaussian noise, original structure gone)
> After training for 200 steps, should we expect perfect protein structures? (Answer: No — our "proteins" are 2D toy shapes; real AF3 trains for weeks)


In [ ]:
# COMPLETE DDPM TRAINING for Protein-Like Structures
# Trains a noise-predicting network and samples new structures

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

# ─── Dataset: 2D "protein backbones" (simplified to 2D for visualization) ─────
def make_protein_shapes(n=500, n_residues=20):
"""Create synthetic 2D 'protein' coordinate data for DDPM demonstration.

Generates simple geometric shapes (circles and line segments) as stand-ins
for protein backbone coordinates — avoids needing real PDB data.

Args:
    n (int): number of synthetic protein samples to generate.
    n_residues (int): number of 2D coordinate points per sample.

Returns:
    Tensor: coordinate array of shape (n, n_residues, 2)."""
    # Generate 3 protein-like motif shapes in 2D
    data = []
    for _ in range(n):
        shape_type = np.random.randint(3)
        t = np.linspace(0, 1, n_residues)
        if shape_type == 0:   # alpha-helix: circular
            x = np.cos(4 * np.pi * t) * 0.8
            y = np.sin(4 * np.pi * t) * 0.8
        elif shape_type == 1: # beta-strand: zigzag
            x = t * 2 - 1
            y = np.sin(n_residues * np.pi * t) * 0.3
        else:                 # random coil: random walk
            dx = np.random.randn(n_residues) * 0.15
            dy = np.random.randn(n_residues) * 0.15
            x = np.cumsum(dx); x -= x.mean()
            y = np.cumsum(dy); y -= y.mean()
        coords = np.stack([x, y], axis=-1)  # (n_residues, 2)
        data.append(coords)
    return torch.tensor(np.array(data), dtype=torch.float32)  # (n, n_residues, 2)

X = make_protein_shapes(n=1000)
print(f"Dataset shape: {X.shape}  (1000 protein shapes, 20 residues, 2D coords)")

# ─── Noise schedule ──────────────────────────────────────────────────────────
T = 300
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)   # alpha_bar_t = prod(1-beta_s for s<=t)

def q_sample(x0, t):
"""Forward diffusion: add noise level t to clean coordinates x0.

Uses the closed-form equation:
    x_t = sqrt(ᾱ_t) · x0 + sqrt(1 − ᾱ_t) · ε,  ε ~ N(0, I)

Args:
    x0 (Tensor): clean data of shape (B, n_residues, 2).
    t (Tensor): integer timestep indices of shape (B,).

Returns:
    Tuple[Tensor, Tensor]: (noisy x_t, noise ε), both shape (B, n_residues, 2)."""
    # Forward process: add noise at timestep t
    # x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * epsilon
    ab = alpha_bars[t].view(-1, 1, 1)
    noise = torch.randn_like(x0)
    return torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise, noise

# ─── Model: noise-predicting UNet (simplified for 2D coordinates) ────────────
class NoisePredictor(nn.Module):
    """UNet-style noise prediction network for protein coordinate DDPM.

    Conditions on continuous noise level t via sinusoidal embedding
    concatenated with flattened coordinate input. Used in the complete
    DDPM training section (Section 9)."""
    def __init__(self, n_res=20, coord_dim=2, hidden=128, t_dim=32):
        """Build the UNet-style noise predictor for protein coordinate diffusion.

Conditions on the continuous noise level via a sinusoidal time embedding
concatenated with the flattened coordinate input.

Args:
    n_res (int): number of residues (spatial positions).
    coord_dim (int): coordinate dimensionality per residue (2 for toy).
    hidden (int): hidden layer width.
    t_dim (int): sinusoidal time embedding dimension."""
        super().__init__()
        self.t_embed = nn.Sequential(
            nn.Linear(1, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim)
        )
        in_dim = n_res * coord_dim + t_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, n_res * coord_dim),
        )
        self.n_res = n_res
        self.coord_dim = coord_dim

    def forward(self, x_t, t):
        """Predict noise in noisy coordinates x_t at timestep t.

Args:
    x_t (Tensor): noisy coordinates of shape (B, n_res, coord_dim).
    t (Tensor): integer timestep indices of shape (B,).

Returns:
    Tensor: predicted noise of shape (B, n_res, coord_dim)."""
        # x_t: (B, n_res, 2), t: (B,) int
        B = x_t.shape[0]
        t_feat = self.t_embed(t.float().unsqueeze(1) / T)  # (B, t_dim)
        x_flat = x_t.reshape(B, -1)                        # (B, n_res*2)
        inp = torch.cat([x_flat, t_feat], dim=-1)
        pred = self.net(inp)
        return pred.reshape(B, self.n_res, self.coord_dim)

model = NoisePredictor()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
print(f"Noise predictor: {sum(p.numel() for p in model.parameters()):,} params")

# ─── Training loop ────────────────────────────────────────────────────────────
losses = []
batch_size = 64
n_steps = 500

for step in range(n_steps):
    idx = torch.randint(0, len(X), (batch_size,))
    x0 = X[idx]
    t = torch.randint(0, T, (batch_size,))

    x_t, noise = q_sample(x0, t)

    pred_noise = model(x_t, t)
    loss = ((pred_noise - noise) ** 2).mean()

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    losses.append(loss.item())
    if step % 100 == 0:
        print(f"  Step {step:4d} | loss = {np.mean(losses[-50:]):.5f}")

# Plot training curve
plt.figure(figsize=(8, 3))
plt.plot(losses, alpha=0.4, color='gray')
window = 20
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(losses)), smoothed, color='steelblue', linewidth=2)
plt.xlabel('Training step'); plt.ylabel('MSE noise prediction loss')
plt.title('DDPM Training Curve'); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ddpm_training.png', dpi=100)
plt.show()

## Reverse Diffusion — Generating Protein Structures from Noise

After the model is trained to predict noise (ε-prediction), generation works by running the reverse Markov chain from pure Gaussian noise back to clean data:

```
x_T ~ N(0, I)
for t = T, T-1, ..., 1:
    ε_pred = model(x_t, t)
    x_{t-1} = (x_t - β_t/√(1-ᾱ_t) · ε_pred) / √α_t  + σ_t · z
```

**Key insight:** the model never sees the final answer during training — it only learns to remove a *small* amount of noise at each step.  Stacking T such steps turns pure noise into a realistic protein-like coordinate cloud.

The code below runs this loop and visualises the denoising trajectory.

In [ ]:
# REVERSE PROCESS: Sample new protein structures from noise
# This is where "generation" happens

@torch.no_grad()
def p_sample(x_t, t_val):
"""One reverse diffusion step: predict and remove noise at timestep t_val.

Implements the ε-prediction formulation:
    x_{t-1} = (1/√α_t) · (x_t − β_t/√(1−ᾱ_t) · ε_θ(x_t, t)) + σ_t·z

Args:
    x_t (Tensor): noisy sample at timestep t_val, shape (B, n_res, 2).
    t_val (int): current integer timestep (counts down from T to 0).

Returns:
    Tensor: denoised sample x_{t-1} of shape (B, n_res, 2)."""
    # One reverse step: predict noise, remove it
    t_batch = torch.full((x_t.shape[0],), t_val, dtype=torch.long)
    pred_noise = model(x_t, t_batch)

    beta_t = betas[t_val]
    alpha_t = alphas[t_val]
    alpha_bar_t = alpha_bars[t_val]

    # DDPM reverse step formula:
    # x_{t-1} = (1/sqrt(alpha_t)) * (x_t - (1-alpha_t)/sqrt(1-alpha_bar_t) * pred_noise)
    #           + sigma_t * z
    mean = (1 / torch.sqrt(alpha_t)) * (
        x_t - (1 - alpha_t) / torch.sqrt(1 - alpha_bar_t) * pred_noise
    )
    if t_val > 0:
        sigma = torch.sqrt(beta_t)
        z = torch.randn_like(x_t)
        return mean + sigma * z
    return mean  # no noise at t=0

@torch.no_grad()
def sample(n_samples=8):
"""Full reverse diffusion: generate protein coordinates from pure Gaussian noise.

Starts from x_T ~ N(0, I) and applies p_sample T times, producing
a batch of final protein-like coordinate arrays.

Args:
    n_samples (int): number of structures to generate (default 8).

Returns:
    Tensor: generated coordinates of shape (n_samples, n_res, 2)."""
    x = torch.randn(n_samples, 20, 2)   # start from pure noise
    for t_val in reversed(range(T)):
        x = p_sample(x, t_val)
    return x

model.eval()
generated = sample(n_samples=9)
print(f"Generated samples shape: {generated.shape}")

# Visualize
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for i, ax in enumerate(axes.flat):
    coords = generated[i].numpy()
    ax.plot(coords[:, 0], coords[:, 1], 'o-', markersize=5, linewidth=1.5, color='steelblue')
    ax.plot(coords[0, 0], coords[0, 1], 'g^', markersize=10, label='N-term')
    ax.plot(coords[-1, 0], coords[-1, 1], 'rs', markersize=10, label='C-term')
    ax.set_title(f'Sample {i+1}'); ax.set_aspect('equal')
    ax.grid(alpha=0.2); ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].legend(fontsize=7)
plt.suptitle('DDPM Generated Protein Shapes (2D Projection)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ddpm_samples.png', dpi=100, bbox_inches='tight')
plt.show()

print()
print("What you just did:")
print("  1. Trained a neural network to predict noise at any timestep t")
print("  2. Started from pure Gaussian noise (step T)")
print("  3. Iteratively denoised 300 steps to generate protein-like shapes")
print()
print("AF3 does this same process but:")
print("  - In 3D with real atom coordinates (not 2D projections)")
print("  - With a Pairformer conditioning the denoising at every step")
print("  - On 200 timesteps with a learned noise schedule")
print("  - On 200M+ training examples from the PDB")

---
## 🎯 Interview Questions — Diffusion Models for Protein Design

**Q1:** What is the fundamental difference between AlphaFold 3 (diffusion for prediction) and RFdiffusion (diffusion for design)?
> **A:** AF3 conditions the diffusion on the sequence — it *predicts* the structure of a given sequence. RFdiffusion conditions on a target motif or binding site — it *designs* a new sequence/structure that satisfies a functional constraint. Both use the same mathematical framework (DDPM), but the conditioning information is completely different.

**Q2:** Why does DDIM (Denoising Diffusion Implicit Models) sample faster than standard DDPM?
> **A:** DDPM requires 1000 denoising steps (Markovian process). DDIM realizes these steps are not actually Markovian — it reformulates to a non-Markovian process that can skip timesteps. In practice: 50 steps instead of 1000 with similar quality. Critical for practical protein design where you want to evaluate thousands of candidates.

**Q3:** What is flow matching and why is it replacing diffusion for protein structure generation in 2024?
> **A:** Flow matching learns a vector field that transports a simple distribution (Gaussian) to the data distribution along straight-line paths. It's simpler to train (no noise schedule), trains faster, and samples in fewer steps. Genie2 and FrameDiff use flow matching and outperform DDPM-based models on protein design benchmarks.

**Q4 (Design thinking):** How would you evaluate whether a diffusion-generated protein structure is worth synthesizing experimentally?
> **A:** (1) AF3 confidence score: pLDDT > 70, pTM > 0.7 for the generated structure (or predict the generated sequence and check AF3's confidence); (2) Rosetta stability score (ΔG folding); (3) AlphaFold-Multimer confidence if it's a binder; (4) ESM-2 pseudo-perplexity of the designed sequence; (5) Check for disulfide bonds / PTMs that can't be expressed in E. coli. Then wet-lab validation.


---
## 🔗 Cross-Module Connections — How Diffusion Relates to Everything Else

Understanding these connections is what separates a **practitioner** from someone who just knows the individual algorithms.

### The Big Picture: What Diffusion Is Doing

Diffusion models learn a **path** from noise (easy to sample from) to data (hard to sample from).
Every other module in this curriculum touches this idea from a different angle:

| Module | Connection to Diffusion | One-Sentence Link |
|--------|------------------------|-------------------|
| **07/05** AlphaFold3 Training | AF3 uses diffusion as its final structure generation step | Pairformer produces pair features → diffusion denoises coordinates |
| **13/01** Bayesian Methods | Both handle uncertainty; diffusion is a variational lower bound | ELBO in VAEs = simplified diffusion ELBO; MC Dropout ≈ noise injection |
| **14/01** Reinforcement Learning | GFlowNets sample proportional to reward — same as diffusion's forward process | GFlowNets can be seen as discrete diffusion on a DAG |
| **15/01** Self-Supervised Learning | Masked token prediction (BERT) is discrete diffusion | Denoising score matching = self-supervised noise prediction |
| **11/02** MD Simulation | Flow matching models (Genie2) learn MD transition paths | AlphaFlow generates conformational ensembles without MD |

### Diffusion vs Flow Matching (2024 state-of-art)

```
Diffusion (DDPM):        x_t = √ᾱ_t · x₀ + √(1-ᾱ_t) · ε   (stochastic path)
Flow Matching (CNF):     dx/dt = v_θ(x, t)                   (deterministic ODE)
```

**Why flow matching is winning for protein design:**
- Straight paths between noise and data → fewer function evaluations at inference
- RFdiffusion 2, Genie2, Boltz-1 all use flow matching or related methods
- Connection to Module 14: the vector field v_θ can be learned with RL-style rewards (RLHF for protein design)

### Debug Checklist: Your Diffusion Model Isn't Working

If samples look like noise (or all collapse to one structure):
1. **Check noise schedule**: β₁ should be ~0.0001, βT should be ~0.02. Too large → everything becomes noise.
2. **Check timestep embedding**: model must see t to denoise correctly. Missing t_embed → all timesteps look the same.
3. **Check loss**: should use predicted noise ε, not directly x₀. Loss = ||ε - ε_θ(x_t, t)||²
4. **Check BF16**: coordinate predictions in BF16 lose precision at small scales; use FP32 for coordinate regression.
5. **Check sampling**: at t=0, clamp predicted x₀ to valid coordinate range.


## 🎤 Interview Questions

**Q1 (Easy):** What is the forward process in a diffusion model?
<details><summary>Answer</summary>
The forward process gradually adds Gaussian noise to data over T timesteps according to a fixed noise schedule. At each step t, a small amount of noise is added: x_t = sqrt(alpha_t) * x_{t-1} + sqrt(1 - alpha_t) * epsilon. After enough steps, the data becomes indistinguishable from pure Gaussian noise.
</details>

**Q2 (Easy):** What does the neural network learn to predict in DDPM?
<details><summary>Answer</summary>
In DDPM, the neural network learns to predict the noise epsilon that was added to the clean data to produce the noisy sample at timestep t. Given x_t and t, the network outputs an estimate of epsilon, and this prediction is used to iteratively denoise from x_T back to x_0 during generation.
</details>

**Q3 (Easy):** What is alpha-bar (ā_t) and why does it let you skip to any noise level?
<details><summary>Answer</summary>
Alpha-bar is the cumulative product of all alpha values from step 1 to t: ā_t = product(alpha_1 ... alpha_t). It enables a closed-form expression x_t = sqrt(ā_t) * x_0 + sqrt(1 - ā_t) * epsilon, which means you can noise the clean data directly to any timestep t without iterating through all intermediate steps. This is essential for efficient training.
</details>

**Q4 (Medium):** Compare DDPM's noise schedule to AF3's variance-exploding schedule.
<details><summary>Answer</summary>
DDPM uses a variance-preserving schedule where noise is added while the signal is scaled down, keeping the total variance roughly constant (controlled by beta_t). AF3 uses a variance-exploding schedule where noise with increasing sigma is added directly to atomic coordinates without rescaling the signal. The VE approach is more natural for 3D coordinates since it preserves the physical scale of the structure while increasing uncertainty, and AF3's noise levels (sigma from 0.01 to 160 angstroms) are calibrated to protein-relevant length scales.
</details>

**Q5 (Medium):** Why does DDPM predict noise epsilon instead of the clean signal x_0?
<details><summary>Answer</summary>
Predicting epsilon leads to a simpler, more stable training objective that reduces to a reweighted variational bound. The noise prediction loss is uniform across noise levels, whereas predicting x_0 directly would require the network to output very different magnitudes at different timesteps. Empirically, epsilon-prediction produces better sample quality. Mathematically, the two parameterisations are equivalent — you can convert between them — but the epsilon form provides better gradient signal during training.
</details>

**Q6 (Medium):** How does classifier-free guidance control generation in protein design?
<details><summary>Answer</summary>
Classifier-free guidance trains a single model both conditionally (with a condition c like a binding motif) and unconditionally (with c dropped out randomly during training). At inference, the guided prediction is: eps_guided = eps_uncond + w * (eps_cond - eps_uncond), where w > 1 amplifies the conditional signal. For protein design, this lets you steer backbone generation toward desired properties (fold, motif, function) with adjustable strength without needing a separate classifier.
</details>

**Q7 (Medium):** What is the relationship between score-based models and diffusion models?
<details><summary>Answer</summary>
Score-based models learn the score function, which is the gradient of the log probability density: nabla_x log p(x). Diffusion models can be reinterpreted as learning the score of the noisy data distribution at each noise level. The connection is formalised through stochastic differential equations (SDEs): the forward diffusion is an SDE, and the reverse generative process uses the learned score to solve the reverse-time SDE. DDPM is a discretisation of this continuous framework, making score-based and diffusion models two views of the same generative paradigm.
</details>

**Q8 (Hard):** Design a conditional diffusion model that generates protein backbones given a binding site motif.
<details><summary>Answer</summary>
Represent the backbone as N, CA, C coordinates per residue. The motif residues (e.g., 5-15 key binding residues with fixed 3D positions) are provided as a condition that remains unnoised throughout diffusion. Architecture: SE(3)-equivariant graph neural network that takes noisy scaffold coordinates + fixed motif coordinates + timestep embedding as input. During training, noise only the non-motif residues. During inference, denoise the scaffold while keeping motif coordinates clamped — this is inpainting-style generation. Add a secondary sequence design module (like ProteinMPNN) to assign amino acids to the generated backbone. Evaluate with designability (inverse folding + AF2 recovery), motif RMSD, and binding energy scores.
</details>

**Q9 (Hard):** How does RFdiffusion combine structure prediction with generative design?
<details><summary>Answer</summary>
RFdiffusion repurposes the RoseTTAFold architecture — originally trained for structure prediction — as the denoising network in a diffusion framework. The key insight is that a structure prediction network already understands protein geometry, so it can be fine-tuned to denoise corrupted structures. The model takes noised 3D coordinates and outputs denoised coordinates, using the same SE(3)-equivariant track architecture. It supports conditional generation through motif scaffolding (fixed residue positions), symmetry constraints, and binder design by conditioning on a target structure. This transfer from prediction to generation is why RFdiffusion produces highly designable backbones.
</details>

**Q10 (Hard):** What are the failure modes of protein diffusion and how would you evaluate generated structures?
<details><summary>Answer</summary>
Key failure modes include: (1) chain breaks and steric clashes from imperfect denoising, (2) mode collapse producing repetitive folds, (3) physically unrealistic Ramachandran angles, (4) generated backbones that no sequence can fold into (low designability). Evaluation pipeline: check bond geometry and clashes (Rosetta score), verify Ramachandran distribution, run inverse folding (ProteinMPNN) to design sequences, fold those sequences with AF2/ESMFold and measure scTM (self-consistency TM-score > 0.5 indicates designable), assess diversity via pairwise TM-score across samples, and for conditional designs verify the motif RMSD is below 1 angstrom. The scTM metric is the gold standard for evaluating protein diffusion models.
</details>

## Notebook Complete

**You can now:**
- Implement the forward noising process and reverse denoising for protein backbone diffusion
- Condition generation on sequence or structural motifs

**Where this fits:**
- Previous: 11_membrane_protein_dynamics
- Next: 02_connecting_modules_12_to_15 or 02_flow_matching_protein_design — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes